# Reconhecimento Inicial — Base SIAPE

Notebook de **reconhecimento estrutural** da base de dados do SIAPE. Este notebook
não analisa o conteúdo dos dados (idade, cargos, distribuições, etc.) — apenas mapeia
a estrutura do arquivo: colunas, tipos, valores ausentes, período coberto por datas e
cardinalidade das colunas categóricas.

**Regra permanente do projeto**: nenhum registro individual (nome, CPF, matrícula,
e-mail, data de nascimento) é exibido em bruto neste notebook. Apenas agregados,
estatísticas e contagens.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/raw/DB_SIAPE_processado_20260729_v2.xlsx")
df = pd.read_excel(DATA_PATH, engine="openpyxl")

## Colunas tratadas como dados pessoais diretos

Estas colunas nunca têm seus valores individuais exibidos neste notebook —
apenas contagens agregadas (ex: quantos valores únicos existem).

In [2]:
COLUNAS_PESSOAIS = [
    "Nome do Servidor",
    "Matrícula SIAPE",
    "CPF",
    "E-Mail Institucional",
    "Data de Nascimento",
]
COLUNAS_PESSOAIS = [c for c in COLUNAS_PESSOAIS if c in df.columns]

print("Colunas tratadas como dados pessoais (nunca exibidas em bruto):")
for c in COLUNAS_PESSOAIS:
    print(f"  - {c}")

Colunas tratadas como dados pessoais (nunca exibidas em bruto):
  - Nome do Servidor
  - Matrícula SIAPE
  - CPF
  - E-Mail Institucional
  - Data de Nascimento


## Dimensões da base

In [3]:
print(f"Número de linhas (servidores): {df.shape[0]}")
print(f"Número de colunas: {df.shape[1]}")

Número de linhas (servidores): 9991
Número de colunas: 62


## Colunas e tipos de dado

In [4]:
tipos = pd.DataFrame({
    "coluna": df.columns,
    "tipo": [str(t) for t in df.dtypes],
})
tipos

,coluna,tipo
0,Cargo,str
1,SIGLA_CARGO,str
2,Nome do Servidor,str
3,Matrícula SIAPE,int64
4,CPF,int64
...,...,...
57,Secretaria/Departamento_Servidores em órgãos f...,str
58,Cidade_Servidores em órgãos fora do DF,str
59,UF_Servidores em órgãos fora do DF,str
60,Tipo de movimentação_Servidores em órgãos fora...,str


## Valores ausentes por coluna

In [5]:
nulos = df.isnull().sum().rename("qtd_nulos").to_frame()
nulos["pct_nulos"] = (nulos["qtd_nulos"] / len(df) * 100).round(1)
nulos.sort_values("qtd_nulos", ascending=False)

,qtd_nulos,pct_nulos
Unnamed: 16,9991,100.0
Cargo,0,0.0
Nome do Servidor,0,0.0
Matrícula SIAPE,0,0.0
CPF,0,0.0
...,...,...
Secretaria/Departamento_Servidores em órgãos fora do DF,0,0.0
Cidade_Servidores em órgãos fora do DF,0,0.0
UF_Servidores em órgãos fora do DF,0,0.0
Tipo de movimentação_Servidores em órgãos fora do DF,0,0.0


## Colunas de data — período coberto

Identifica colunas cujo nome sugere data e tenta converter os valores.
Mostra apenas a data mínima e máxima encontradas (agregado), nunca os valores linha a linha.

In [6]:
colunas_data_candidatas = [c for c in df.columns if "data" in c.lower()]

resumo_datas = []
for col in colunas_data_candidatas:
    convertida = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
    validos = int(convertida.notna().sum())
    if validos == 0:
        continue
    resumo_datas.append({
        "coluna": col,
        "valores_validos": validos,
        "valores_nao_convertidos": int(df[col].notna().sum()) - validos,
        "data_minima": convertida.min(),
        "data_maxima": convertida.max(),
    })

resumo_datas_df = pd.DataFrame(resumo_datas)
resumo_datas_df

C:\Users\dutra\AppData\Local\Temp\ipykernel_15000\806259804.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  convertida = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
C:\Users\dutra\AppData\Local\Temp\ipykernel_15000\806259804.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  convertida = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
C:\Users\dutra\AppData\Local\Temp\ipykernel_15000\806259804.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  convertida = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
C:\Users\dutra\AppData\Local\Temp\ipy

C:\Users\dutra\AppData\Local\Temp\ipykernel_15000\806259804.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  convertida = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
C:\Users\dutra\AppData\Local\Temp\ipykernel_15000\806259804.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  convertida = pd.to_datetime(df[col], errors="coerce", dayfirst=True)


,coluna,valores_validos,valores_nao_convertidos,data_minima,data_maxima
0,Data de Nascimento,4599,5392,1942-05-21,2002-09-08
1,Data de Ingresso no Órgão,9991,0,2006-07-14,2026-07-10
2,Data de início_Afastados para estudo,29,9962,2022-09-19,2026-03-13
3,Data estimada de encerramento_Afastados para e...,29,9962,2026-01-07,2030-12-03
4,Data de início_DB_SGC,2329,7662,1999-02-07,2103-04-07
5,Ingresso na carreira_Datas de ingresso,1626,8365,1990-01-02,2023-11-05
6,Ingresso no órgão atual_Datas de ingresso,2329,7662,1999-02-07,2103-04-07
7,Data de início_Licenciados,105,9886,2001-04-30,2026-08-05
8,Data estimada de encerramento_Licenciados,92,9899,2025-09-16,2029-08-05


## Cardinalidade das colunas categóricas

Quantidade de valores únicos por coluna de texto. Os valores em si não são listados
— apenas a contagem — especialmente para colunas de dados pessoais.

In [7]:
colunas_categoricas = df.select_dtypes(include=["object"]).columns.tolist()

cardinalidade = pd.DataFrame({
    "coluna": colunas_categoricas,
    "valores_unicos": [int(df[c].nunique(dropna=True)) for c in colunas_categoricas],
    "dado_pessoal": [c in COLUNAS_PESSOAIS for c in colunas_categoricas],
}).sort_values("valores_unicos", ascending=False).reset_index(drop=True)

cardinalidade

C:\Users\dutra\AppData\Local\Temp\ipykernel_15000\3336661157.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_categoricas = df.select_dtypes(include=["object"]).columns.tolist()


,coluna,valores_unicos,dado_pessoal
0,Nome do Servidor,9988,True
1,E-Mail Institucional,9656,True
2,Data de Nascimento,8202,True
3,Unidade_DB_SGC,1206,False
4,Data de início_DB_SGC,1144,False
5,Ingresso no órgão atual_Datas de ingresso,1144,False
6,Unidade de Exercício (Origem),661,False
7,Ingresso na carreira_Datas de ingresso,411,False
8,Secretaria/Departamento_DB_SGC,388,False
9,Órgão de Exercício,143,False


## Amostra estrutural (sem colunas pessoais)

Exibe as primeiras linhas apenas das colunas estruturais, após remover todas as
colunas de identificação direta (nome, CPF, matrícula, e-mail, data de nascimento).

In [8]:
df.drop(columns=COLUNAS_PESSOAIS).head()

,Cargo,SIGLA_CARGO,Idade,Sexo,Escolaridade,Data de Ingresso no Órgão,UF,Unidade de Exercício (Origem),Secretaria,Órgão de Exercício,...,UF_Licenciados,País_Licenciados,Data de início_Licenciados,Data estimada de encerramento_Licenciados,Órgão/Entidade_Servidores em órgãos fora do DF,Secretaria/Departamento_Servidores em órgãos fora do DF,Cidade_Servidores em órgãos fora do DF,UF_Servidores em órgãos fora do DF,Tipo de movimentação_Servidores em órgãos fora do DF,Tipo de movimentação_Situação atual por tipo de movimentação
0,ADMINISTRADOR,ATE,72,FEMININO,ENSINO SUPERIOR,2023-08-01,DF,DIV DE PAG DE EXERC ANT,SSC,MIN GESTAO E INOV EM SERV PUBLICOS,...,NI,NI,NI,NI,NI,NI,NI,NI,NI,NI
1,ADMINISTRADOR,ATE,68,FEMININO,ENSINO SUPERIOR,2025-08-01,DF,UORG SERVIDORES ADM ART. 214 - L15141/25,MGI/SE,MINISTERIO DA AGRICULTURA E PECUARIA,...,NI,NI,NI,NI,NI,NI,NI,NI,NI,NI
2,ADMINISTRADOR,ATE,69,FEMININO,ENSINO SUPERIOR,2023-08-01,DF,DIVISAO DE PROGRAMACAO FINANCEIRA,SSC,MIN GESTAO E INOV EM SERV PUBLICOS,...,NI,NI,NI,NI,NI,NI,NI,NI,NI,NI
3,ADMINISTRADOR,ATE,66,MASCULINO,ENSINO SUPERIOR,2025-09-01,DF,UORG SERVIDORES ADM ART. 214 - L15141/25,MGI/SE,MINISTERIO DA FAZENDA,...,NI,NI,NI,NI,NI,NI,NI,NI,NI,NI
4,ADMINISTRADOR,ATE,73,MASCULINO,ENSINO SUPERIOR,2025-09-01,DF,UORG SERVIDORES ADM ART. 214 - L15141/25,MGI/SE,MINISTERIO DA FAZENDA,...,NI,NI,NI,NI,NI,NI,NI,NI,NI,NI


## Resumo consolidado

In [9]:
from IPython.display import Markdown, display

n_linhas, n_colunas = df.shape
n_nulos_total = int(df.isnull().sum().sum())
colunas_com_nulos = int((nulos["qtd_nulos"] > 0).sum())

if not resumo_datas_df.empty:
    data_min_geral = resumo_datas_df["data_minima"].min()
    data_max_geral = resumo_datas_df["data_maxima"].max()
    texto_datas = (
        f"- Foram encontradas **{len(resumo_datas_df)}** colunas de data com valores validos, "
        f"cobrindo o periodo de **{data_min_geral:%d/%m/%Y}** a **{data_max_geral:%d/%m/%Y}**."
    )
else:
    texto_datas = "- Nenhuma coluna de data com valores validos foi encontrada."

resumo_md = f"""
A base contem **{n_linhas} linhas** (servidores) e **{n_colunas} colunas**.

- **{colunas_com_nulos}** colunas possuem ao menos um valor ausente, totalizando **{n_nulos_total}** valores nulos na base inteira.
{texto_datas}
- Foram identificadas **{len(colunas_categoricas)}** colunas categoricas (texto), com cardinalidade variando de **{cardinalidade["valores_unicos"].min()}** a **{cardinalidade["valores_unicos"].max()}** valores unicos.
- **{len(COLUNAS_PESSOAIS)}** colunas foram tratadas como dados pessoais diretos e nao tiveram nenhum valor individual exibido: {", ".join(COLUNAS_PESSOAIS)}.

Nenhum registro individual foi exibido neste notebook, exceto a amostra estrutural acima, da qual todas as colunas de identificacao direta foram removidas.
"""

display(Markdown(resumo_md))


A base contem **9991 linhas** (servidores) e **62 colunas**.

- **1** colunas possuem ao menos um valor ausente, totalizando **9991** valores nulos na base inteira.
- Foram encontradas **9** colunas de data com valores validos, cobrindo o periodo de **21/05/1942** a **07/04/2103**.
- Foram identificadas **56** colunas categoricas (texto), com cardinalidade variando de **2** a **9988** valores unicos.
- **5** colunas foram tratadas como dados pessoais diretos e nao tiveram nenhum valor individual exibido: Nome do Servidor, Matrícula SIAPE, CPF, E-Mail Institucional, Data de Nascimento.

Nenhum registro individual foi exibido neste notebook, exceto a amostra estrutural acima, da qual todas as colunas de identificacao direta foram removidas.
